In [22]:
import pandas as pd
import numpy as np

import cogent3
from cogent3 import get_app
from collections import Counter
from cogent3 import load_unaligned_seqs


import paths
import libs

motif_length = 3

REGIONS = ["cds", "introns_nonUTR", "introns3UTR", "introns5UTR", "intronsAR", "distalIG", "proximal5IG", "proximal3IG"]
CHROMOSOMES = [str(i) for i in range(1, 23)] + ["X"]

In [23]:
def get_Oman_mutability_model():
    # Oman et al. 2022 mutability model doi.org/10.1093/gbe/evac032
    #https://github.com/MadeleineOman/Mutation_equilibrium/blob/master/Human_mutability_model/Model_2020_12_02_genomeWide.txt
    Mut_model = {"TGC": [2.9972579195868547e-05, {"TAC": 0.6179604261796042, "TTC": 0.24429223744292236, "TCC": 0.13774733637747336}], "GCG": [0.00029305478867835234, {"GGG": 0.024635036496350366, "GTG": 0.9023722627737226, "GAG": 0.072992700729927}], "CGG": [0.0002844044368929987, {"CTG": 0.008481421647819063, "CAG": 0.9483037156704361, "CCG": 0.04321486268174475}], "AGA": [2.9567248890908222e-05, {"ACA": 0.3502487562189055, "AAA": 0.47512437810945274, "ATA": 0.1746268656716418}], "GTT": [2.080387143493786e-05, {"GGT": 0.17634408602150536, "GAT": 0.15483870967741936, "GCT": 0.6688172043010753}], "TTG": [1.670667298920289e-05, {"TCG": 0.564729867482161, "TAG": 0.1600407747196738, "TGG": 0.27522935779816515}], "GTG": [2.072125413895217e-05, {"GGG": 0.19833852544132918, "GCG": 0.6261682242990654, "GAG": 0.1754932502596054}], "ATA": [3.830079226523379e-05, {"ACA": 0.8031066330814441, "AAA": 0.12342569269521411, "AGA": 0.07346767422334173}], "TAA": [1.9926310254932553e-05, {"TTA": 0.21774193548387097, "TGA": 0.5669354838709677, "TCA": 0.2153225806451613}], "TGA": [2.5483876991879155e-05, {"TTA": 0.12442698100851342, "TCA": 0.2547478716437459, "TAA": 0.6208251473477406}], "ACC": [4.907736401728898e-05, {"ATC": 0.5266203703703703, "AGC": 0.1550925925925926, "AAC": 0.31828703703703703}], "AAT": [3.072691806317887e-05, {"AGT": 0.7734375, "ACT": 0.11328125, "ATT": 0.11328125}], "ATT": [2.9485873550591224e-05, {"AGT": 0.1134020618556701, "ACT": 0.7445091887046168, "AAT": 0.14208874943971314}], "TAC": [2.3436542239015503e-05, {"TGC": 0.7164366373902133, "TTC": 0.1480552070263488, "TCC": 0.1355081555834379}], "AGC": [3.287185460804443e-05, {"ATC": 0.15373665480427046, "ACC": 0.21281138790035586, "AAC": 0.6334519572953736}], "CCG": [0.00028532665311129646, {"CAG": 0.01155115511551155, "CGG": 0.05033003300330033, "CTG": 0.9381188118811881}], "CCT": [3.304925392418301e-05, {"CAT": 0.10123042505592841, "CTT": 0.6582774049217002, "CGT": 0.24049217002237136}], "AAG": [1.6973322432504848e-05, {"ACG": 0.22395326192794549, "ATG": 0.16747809152872445, "AGG": 0.6085686465433301}], "TTC": [1.3849480015946078e-05, {"TGC": 0.2331360946745562, "TAC": 0.2, "TCC": 0.5668639053254438}], "CCC": [3.725928520944783e-05, {"CAC": 0.11978465679676985, "CTC": 0.6440107671601615, "CGC": 0.23620457604306863}], "TCA": [2.5025871746211233e-05, {"TTA": 0.6286666666666667, "TGA": 0.242, "TAA": 0.12933333333333333}], "GCT": [3.203685050542384e-05, {"GTT": 0.637905604719764, "GAT": 0.1733038348082596, "GGT": 0.1887905604719764}], "ACT": [3.561156290160249e-05, {"AGT": 0.24214734437464305, "AAT": 0.1559109080525414, "ATT": 0.6019417475728155}], "GAC": [1.747618852010978e-05, {"GTC": 0.25544554455445545, "GCC": 0.16831683168316833, "GGC": 0.5762376237623762}], "ATC": [2.5312982945069347e-05, {"ACC": 0.5769980506822612, "AGC": 0.11208576998050682, "AAC": 0.310916179337232}], "ACA": [4.0115984139892814e-05, {"AAA": 0.1690312119983786, "ATA": 0.6558573165788407, "AGA": 0.17511147142278072}], "ATG": [4.061850801861585e-05, {"ACG": 0.720620842572062, "AAG": 0.15121951219512195, "AGG": 0.12815964523281598}], "GCC": [3.932724198051991e-05, {"GAC": 0.23960535588442566, "GTC": 0.6102889358703312, "GGC": 0.15010570824524314}], "TTT": [1.490776574686971e-05, {"TAT": 0.18596691386195094, "TCT": 0.5236737022247575, "TGT": 0.2903593839132915}], "CAA": [1.789369737451358e-05, {"CTA": 0.11730205278592376, "CGA": 0.5884652981427175, "CCA": 0.29423264907135877}], "CTC": [1.4898688915375447e-05, {"CAC": 0.21586475942782835, "CGC": 0.21976592977893367, "CCC": 0.5643693107932379}], "CAG": [2.1340608009646854e-05, {"CTG": 0.14135338345864662, "CGG": 0.6466165413533834, "CCG": 0.21203007518796993}], "TCT": [2.8974834575852665e-05, {"TTT": 0.4771341463414634, "TAT": 0.17682926829268292, "TGT": 0.34603658536585363}], "CCA": [2.6093855314575102e-05, {"CTA": 0.7006896551724138, "CAA": 0.09517241379310344, "CGA": 0.20413793103448277}], "AGG": [3.145279418412814e-05, {"ACG": 0.21803182086034179, "ATG": 0.10724808485562758, "AAG": 0.6747200942840307}], "TTA": [2.052268059289879e-05, {"TGA": 0.19560094265514533, "TCA": 0.5687352710133543, "TAA": 0.2356637863315004}], "CAC": [2.022959103483819e-05, {"CTC": 0.1896551724137931, "CGC": 0.6012931034482759, "CCC": 0.20905172413793102}], "CGC": [0.0002917845917779848, {"CAC": 0.9085027726432532, "CTC": 0.06099815157116451, "CCC": 0.030499075785582256}], "CGT": [0.00035917531250873904, {"CAT": 0.9214459506430309, "CTT": 0.0413625304136253, "CCT": 0.037191518943343764}], "TCC": [3.2918218175037877e-05, {"TGC": 0.23849643551523006, "TAC": 0.15294880103694103, "TTC": 0.6085547634478289}], "CGA": [0.00026243687166486475, {"CTA": 0.022259321090706732, "CAA": 0.9237618252643295, "CCA": 0.053978853644963826}], "GGC": [3.889942950368404e-05, {"GAC": 0.586600142551675, "GTC": 0.23734853884533144, "GCC": 0.17605131860299358}], "GTA": [2.3578808402033628e-05, {"GCA": 0.7385943279901356, "GAA": 0.14303329223181258, "GGA": 0.11837237977805179}], "TAG": [1.824544093318805e-05, {"TTG": 0.1424581005586592, "TCG": 0.15921787709497207, "TGG": 0.6983240223463687}], "AAC": [1.968746908573502e-05, {"ATC": 0.15107102593010147, "ACC": 0.17925591882750846, "AGC": 0.6696730552423901}], "CAT": [4.251614829783026e-05, {"CTT": 0.1483354403708386, "CCT": 0.13864306784660768, "CGT": 0.7130214917825537}], "GCA": [3.2595686902491635e-05, {"GTA": 0.5868055555555556, "GAA": 0.24513888888888888, "GGA": 0.16805555555555557}], "CTT": [1.7428227261803167e-05, {"CAT": 0.1383177570093458, "CCT": 0.6177570093457944, "CGT": 0.24392523364485982}], "AGT": [3.6550914772674284e-05, {"ACT": 0.2545961002785515, "AAT": 0.5910863509749303, "ATT": 0.1543175487465181}], "GAA": [1.3198444035923253e-05, {"GTA": 0.1982651796778191, "GCA": 0.24535315985130113, "GGA": 0.5563816604708798}], "TGG": [2.5882067388923472e-05, {"TTG": 0.10845839017735334, "TCG": 0.21145975443383355, "TAG": 0.680081855388813}], "GGA": [3.248349026607227e-05, {"GTA": 0.18717948717948718, "GCA": 0.2153846153846154, "GAA": 0.5974358974358974}], "TGT": [4.0750169729063745e-05, {"TTT": 0.1638238794129314, "TAT": 0.6552955176517256, "TCT": 0.1808806029353431}], "GGG": [3.3624356093580806e-05, {"GTG": 0.11970260223048328, "GCG": 0.2342007434944238, "GAG": 0.6460966542750929}], "ACG": [0.00036217158636308954, {"ATG": 0.9236376258243666, "AAG": 0.04547032280458174, "AGG": 0.030892051371051717}], "GAG": [1.4676934848815485e-05, {"GGG": 0.541501976284585, "GTG": 0.2450592885375494, "GCG": 0.2134387351778656}], "CTG": [2.1833439706371427e-05, {"CAG": 0.15476190476190477, "CGG": 0.20610119047619047, "CCG": 0.6391369047619048}], "CTA": [2.0923797199369e-05, {"CAA": 0.13496932515337423, "CCA": 0.694478527607362, "CGA": 0.1705521472392638}], "GAT": [2.4013326542965934e-05, {"GTT": 0.283248730964467, "GGT": 0.5959390862944163, "GCT": 0.12081218274111676}], "GTC": [2.0404272934263023e-05, {"GAC": 0.2345890410958904, "GCC": 0.5856164383561644, "GGC": 0.1797945205479452}], "GGT": [4.9361488694784825e-05, {"GTT": 0.311277330264672, "GAT": 0.5385500575373993, "GCT": 0.15017261219792866}], "TCG": [0.0002461141623593142, {"TTG": 0.9181556195965418, "TAG": 0.03170028818443804, "TGG": 0.05014409221902017}], "TAT": [3.9701506518359236e-05, {"TTT": 0.11926977687626775, "TCT": 0.0795131845841785, "TGT": 0.8012170385395537}], "AAA": [1.4524195153728667e-05, {"ACA": 0.30213270142180093, "ATA": 0.1735781990521327, "AGA": 0.5242890995260664}]}
    #extract the first value of the Mut_model. t
    # This selects the total relative rate of lets say TGC witouth considering whether the mutaiton is TAC, TTC or TCC
    Ind_mut = {k: t[0] for k, t in Mut_model.items()}
    average_mut = sum(Ind_mut.values())/len(Ind_mut)

    #the model is a relative mutation rate. This means that I can divide or multiply by any constant and the model is the same.
    #To avoid numerical error, I divide all elements by the average mutability value in the Oman model
    Prop_model = {k: v / average_mut for k, v in Ind_mut.items()}
    Prop_model = Counter(Prop_model)

    return Prop_model

In [ ]:
mutability_model = get_Oman_mutability_model()

row_data_mutability = []
for chromosome in CHROMOSOMES:
    for region in REGIONS:
        region_path = region + "/chrm" + chromosome
        folder_in = paths.DATA_HUMCHIMPORANGOR114 + region_path
        file_in = folder_in + "/human_seq.fa"

        humanseq = load_unaligned_seqs(file_in, moltype="dna")
        humanseq = humanseq.seqs["Human"]

        #Making a table of motifs and how many times they occur
        table_motifs = libs.number_of_motifs(motif_length=motif_length)
        motifs_Counter = table_motifs(humanseq)

        # Multiplying motifs by relative mutation
        sitesTimesMutability = Counter(
            {k: motifs_Counter[k] * mutability_model[k] for k in motifs_Counter.keys() & mutability_model.keys()}
        )

        numb_motifs = sum(motifs_Counter.values())

        #Divide the total relative mutability sum over the number of motifs considered to get an average mutability score
        mean_mutability = sum(sitesTimesMutability.values())/numb_motifs

        row_data_mutability.append({
            "Region": region,
            "Chromosome": chromosome,
            "mean_mutability": mean_mutability,
            "numb_sites": len(humanseq)
        })

file_out = "output_data/mutability_motiflength" + str(motif_length) + ".csv"
pd.DataFrame(row_data_mutability).to_csv(file_out, index=False)


In [25]:
pd.DataFrame(row_data_mutability)

,Region,Chromosome,mean_mutability,numb_sites
0,cds,1,0.716698,3267138
1,introns_nonUTR,1,0.519656,55059024
2,introns3UTR,1,0.669294,650415
3,introns5UTR,1,0.673571,689064
4,intronsAR,1,0.535451,116436765
...,...,...,...,...
179,introns5UTR,X,0.632280,259638
180,intronsAR,X,0.517126,52711425
181,distalIG,X,0.487814,51798978
182,proximal5IG,X,0.521340,5427300
